# Train and run an iris classifier, all in Go

Train a tiny neural network on Fisher's iris dataset with hand-written
backprop/Adam, write it out as a TensorFlow Lite flatbuffer, and run it with
go-tflite — no Python, no TensorFlow. Adapted from
[go-iris-tflite](https://github.com/mattn/go-iris-tflite).

In [ ]:
import (
    "fmt"
    "io"
    "net/http"
    "os"
)
os.MkdirAll("data", 0755)
resp, err := http.Get("https://raw.githubusercontent.com/mattn/go-iris-tflite/main/data/iris.csv")
if err != nil { panic(err) }
f, _ := os.Create("data/iris.csv")
io.Copy(f, resp.Body)
f.Close()
resp.Body.Close()

In [ ]:
import "github.com/mattn/go-iris-tflite/iris"
x, y, err := iris.Load("data/iris.csv")
fmt.Println(len(x), "flowers", err)

## The model

`dense(4→8) tanh, dense(8→3) softmax` — 91 parameters, trained with Adam.

In [ ]:
import (
    "math"
    "math/rand"
)

const (
    inputs  = iris.Dim
    hidden  = 8
    classes = 3

    epochs    = 400
    batchSize = 16

    lr      = 0.001
    beta1   = 0.9
    beta2   = 0.999
    epsilon = 1e-7
)

type stats struct {
    mean, std [inputs]float64
}

func newStats(x [][inputs]float64) stats {
    var s stats
    for _, v := range x {
        for i, f := range v {
            s.mean[i] += f
        }
    }
    for i := range s.mean {
        s.mean[i] /= float64(len(x))
    }
    for _, v := range x {
        for i, f := range v {
            d := f - s.mean[i]
            s.std[i] += d * d
        }
    }
    for i := range s.std {
        s.std[i] = math.Sqrt(s.std[i] / float64(len(x)))
    }
    return s
}

func (s stats) apply(v [inputs]float64) []float64 {
    out := make([]float64, inputs)
    for i, f := range v {
        out[i] = (f - s.mean[i]) / s.std[i]
    }
    return out
}

type dense struct {
    in, out        int
    w, b           []float64
    mw, vw, mb, vb []float64
}

func newDense(in, out int, r *rand.Rand) *dense {
    d := &dense{
        in: in, out: out,
        w: make([]float64, in*out), b: make([]float64, out),
        mw: make([]float64, in*out), vw: make([]float64, in*out),
        mb: make([]float64, out), vb: make([]float64, out),
    }
    limit := math.Sqrt(6.0 / float64(in+out))
    for i := range d.w {
        d.w[i] = (r.Float64()*2 - 1) * limit
    }
    return d
}

func (d *dense) apply(gw, gb []float64, t int) {
    alpha := lr * math.Sqrt(1-math.Pow(beta2, float64(t))) / (1 - math.Pow(beta1, float64(t)))
    adam := func(w, m, v, g []float64) {
        for i, gi := range g {
            m[i] = beta1*m[i] + (1-beta1)*gi
            v[i] = beta2*v[i] + (1-beta2)*gi*gi
            w[i] -= alpha * m[i] / (math.Sqrt(v[i]) + epsilon)
        }
    }
    adam(d.w, d.mw, d.vw, gw)
    adam(d.b, d.mb, d.vb, gb)
}

In [ ]:
func forward(l1, l2 *dense, x []float64, h *[hidden]float64, p *[classes]float64) {
    copy(h[:], l1.b)
    for i, v := range x {
        w := l1.w[i*hidden : (i+1)*hidden]
        for j := range h {
            h[j] += v * w[j]
        }
    }
    for j := range h {
        h[j] = math.Tanh(h[j])
    }

    copy(p[:], l2.b)
    for j := range h {
        w := l2.w[j*classes : (j+1)*classes]
        for k := range p {
            p[k] += h[j] * w[k]
        }
    }
    max := p[0]
    for _, v := range p[1:] {
        if v > max {
            max = v
        }
    }
    var sum float64
    for k := range p {
        p[k] = math.Exp(p[k] - max)
        sum += p[k]
    }
    for k := range p {
        p[k] /= sum
    }
}

type grad struct {
    w1, b1, w2, b2 []float64
}

func zero(v []float64) {
    for i := range v {
        v[i] = 0
    }
}

func step(l1, l2 *dense, x, y []float64, invN float64, g *grad) {
    var h [hidden]float64
    var p [classes]float64
    forward(l1, l2, x, &h, &p)

    var d2 [classes]float64
    for k := range d2 {
        d2[k] = (p[k] - y[k]) * invN
        g.b2[k] += d2[k]
    }

    var d1 [hidden]float64
    for j := range h {
        w := l2.w[j*classes : (j+1)*classes]
        gw := g.w2[j*classes : (j+1)*classes]
        var dh float64
        for k, d := range d2 {
            gw[k] += h[j] * d
            dh += w[k] * d
        }
        d1[j] = dh * (1 - h[j]*h[j])
        g.b1[j] += d1[j]
    }
    for i, v := range x {
        gw := g.w1[i*hidden : (i+1)*hidden]
        for j := range d1 {
            gw[j] += v * d1[j]
        }
    }
}

func train(r *rand.Rand, trX [][]float64, trY []int) (*dense, *dense) {
    l1 := newDense(inputs, hidden, r)
    l2 := newDense(hidden, classes, r)
    onehot := make([][]float64, len(trY))
    for i, label := range trY {
        onehot[i] = make([]float64, classes)
        onehot[i][label] = 1
    }
    idx := make([]int, len(trX))
    for i := range idx {
        idx[i] = i
    }
    g := &grad{
        w1: make([]float64, inputs*hidden),
        b1: make([]float64, hidden),
        w2: make([]float64, hidden*classes),
        b2: make([]float64, classes),
    }
    t := 0
    for epoch := 1; epoch <= epochs; epoch++ {
        r.Shuffle(len(idx), func(i, j int) { idx[i], idx[j] = idx[j], idx[i] })
        for lo := 0; lo < len(idx); lo += batchSize {
            hi := lo + batchSize
            if hi > len(idx) {
                hi = len(idx)
            }
            invN := 1 / float64(hi-lo)
            zero(g.w1)
            zero(g.b1)
            zero(g.w2)
            zero(g.b2)
            for _, i := range idx[lo:hi] {
                step(l1, l2, trX[i], onehot[i], invN, g)
            }
            t++
            l2.apply(g.w2, g.b2, t)
            l1.apply(g.w1, g.b1, t)
        }
    }
    return l1, l2
}

func argmax(v []float64) int {
    best := 0
    for i, x := range v {
        if x > v[best] {
            best = i
        }
    }
    return best
}

## Train on all 150 flowers

In [ ]:
import "time"

s := newStats(x)
trX := make([][]float64, len(x))
for i, v := range x {
    trX[i] = s.apply(v)
}
started := time.Now()
l1, l2 := train(rand.New(rand.NewSource(1)), trX, y)
ok := 0
for i := range x {
    var h [hidden]float64
    var p [classes]float64
    forward(l1, l2, trX[i], &h, &p)
    if argmax(p[:]) == y[i] {
        ok++
    }
}
fmt.Printf("%d/%d on the training data (%v)\n", ok, len(x), time.Since(started).Round(time.Millisecond))

// Fold the standardization into layer 1 so the model takes raw centimeters.
for i := 0; i < l1.in; i++ {
    w := l1.w[i*l1.out : (i+1)*l1.out]
    for j := range w {
        w[j] /= s.std[i]
        l1.b[j] -= s.mean[i] * w[j]
    }
}

## Write the model as a TensorFlow Lite flatbuffer

The schema is small enough to emit by hand (see
`tensorflow/lite/schema/schema.fbs`).

In [ ]:
import (
    "encoding/binary"
    flatbuffers "github.com/google/flatbuffers/go"
)

const (
    opFullyConnected = 9
    opSoftmax        = 25
    opTanh           = 28

    optionsFullyConnected = 8
    optionsSoftmax        = 9
)

func floatBytes(v []float64) []byte {
    b := make([]byte, 4*len(v))
    for i, x := range v {
        binary.LittleEndian.PutUint32(b[i*4:], math.Float32bits(float32(x)))
    }
    return b
}

func transposed(d *dense) []float64 {
    t := make([]float64, len(d.w))
    for i := 0; i < d.in; i++ {
        for j := 0; j < d.out; j++ {
            t[j*d.in+i] = d.w[i*d.out+j]
        }
    }
    return t
}

func fbIntVector(b *flatbuffers.Builder, vals []int32) flatbuffers.UOffsetT {
    b.StartVector(4, len(vals), 4)
    for i := len(vals) - 1; i >= 0; i-- {
        b.PrependInt32(vals[i])
    }
    return b.EndVector(len(vals))
}

func fbOffsetVector(b *flatbuffers.Builder, offs []flatbuffers.UOffsetT) flatbuffers.UOffsetT {
    b.StartVector(4, len(offs), 4)
    for i := len(offs) - 1; i >= 0; i-- {
        b.PrependUOffsetT(offs[i])
    }
    return b.EndVector(len(offs))
}

func fbBuffer(b *flatbuffers.Builder, data []byte) flatbuffers.UOffsetT {
    var off flatbuffers.UOffsetT
    if len(data) > 0 {
        b.Prep(16, len(data))
        off = b.CreateByteVector(data)
    }
    b.StartObject(1)
    if len(data) > 0 {
        b.PrependUOffsetTSlot(0, off, 0)
    }
    return b.EndObject()
}

func fbTensor(b *flatbuffers.Builder, name string, shape []int32, buffer int32) flatbuffers.UOffsetT {
    nameOff := b.CreateString(name)
    shapeOff := fbIntVector(b, shape)
    b.StartObject(4)
    b.PrependUOffsetTSlot(0, shapeOff, 0)
    b.PrependUint32Slot(2, uint32(buffer), 0)
    b.PrependUOffsetTSlot(3, nameOff, 0)
    return b.EndObject()
}

func fbOperatorCode(b *flatbuffers.Builder, code int32) flatbuffers.UOffsetT {
    b.StartObject(4)
    b.PrependInt8Slot(0, int8(code), 0)
    b.PrependInt32Slot(3, code, 0)
    return b.EndObject()
}

func fbOperator(b *flatbuffers.Builder, opcodeIndex uint32, ins, outs []int32, optionsType byte, options flatbuffers.UOffsetT) flatbuffers.UOffsetT {
    inOff := fbIntVector(b, ins)
    outOff := fbIntVector(b, outs)
    b.StartObject(5)
    b.PrependUint32Slot(0, opcodeIndex, 0)
    b.PrependUOffsetTSlot(1, inOff, 0)
    b.PrependUOffsetTSlot(2, outOff, 0)
    if optionsType != 0 {
        b.PrependByteSlot(3, optionsType, 0)
        b.PrependUOffsetTSlot(4, options, 0)
    }
    return b.EndObject()
}

func buildTFLite(l1, l2 *dense) []byte {
    b := flatbuffers.NewBuilder(16 * 1024)

    buffers := []flatbuffers.UOffsetT{
        fbBuffer(b, nil),
        fbBuffer(b, floatBytes(transposed(l1))),
        fbBuffer(b, floatBytes(l1.b)),
        fbBuffer(b, floatBytes(transposed(l2))),
        fbBuffer(b, floatBytes(l2.b)),
    }

    tensors := []flatbuffers.UOffsetT{
        fbTensor(b, "input", []int32{1, inputs}, 0),
        fbTensor(b, "dense/kernel", []int32{hidden, inputs}, 1),
        fbTensor(b, "dense/bias", []int32{hidden}, 2),
        fbTensor(b, "dense/BiasAdd", []int32{1, hidden}, 0),
        fbTensor(b, "tanh", []int32{1, hidden}, 0),
        fbTensor(b, "dense_1/kernel", []int32{classes, hidden}, 3),
        fbTensor(b, "dense_1/bias", []int32{classes}, 4),
        fbTensor(b, "dense_1/BiasAdd", []int32{1, classes}, 0),
        fbTensor(b, "output", []int32{1, classes}, 0),
    }

    opcodes := []flatbuffers.UOffsetT{
        fbOperatorCode(b, opFullyConnected),
        fbOperatorCode(b, opTanh),
        fbOperatorCode(b, opSoftmax),
    }

    b.StartObject(4)
    fcOpts1 := b.EndObject()
    b.StartObject(4)
    fcOpts2 := b.EndObject()
    b.StartObject(1)
    b.PrependFloat32Slot(0, 1.0, 0.0)
    smOpts := b.EndObject()

    operators := []flatbuffers.UOffsetT{
        fbOperator(b, 0, []int32{0, 1, 2}, []int32{3}, optionsFullyConnected, fcOpts1),
        fbOperator(b, 1, []int32{3}, []int32{4}, 0, 0),
        fbOperator(b, 0, []int32{4, 5, 6}, []int32{7}, optionsFullyConnected, fcOpts2),
        fbOperator(b, 2, []int32{7}, []int32{8}, optionsSoftmax, smOpts),
    }

    subgraphName := b.CreateString("main")
    tensorsOff := fbOffsetVector(b, tensors)
    inputsOff := fbIntVector(b, []int32{0})
    outputsOff := fbIntVector(b, []int32{8})
    operatorsOff := fbOffsetVector(b, operators)
    b.StartObject(5)
    b.PrependUOffsetTSlot(0, tensorsOff, 0)
    b.PrependUOffsetTSlot(1, inputsOff, 0)
    b.PrependUOffsetTSlot(2, outputsOff, 0)
    b.PrependUOffsetTSlot(3, operatorsOff, 0)
    b.PrependUOffsetTSlot(4, subgraphName, 0)
    subgraph := b.EndObject()

    description := b.CreateString("Iris species classifier, trained in pure Go.")
    opcodesOff := fbOffsetVector(b, opcodes)
    subgraphsOff := fbOffsetVector(b, []flatbuffers.UOffsetT{subgraph})
    buffersOff := fbOffsetVector(b, buffers)
    b.StartObject(5)
    b.PrependUint32Slot(0, 3, 0)
    b.PrependUOffsetTSlot(1, opcodesOff, 0)
    b.PrependUOffsetTSlot(2, subgraphsOff, 0)
    b.PrependUOffsetTSlot(3, description, 0)
    b.PrependUOffsetTSlot(4, buffersOff, 0)
    model := b.EndObject()

    b.FinishWithFileIdentifier(model, []byte("TFL3"))
    return b.FinishedBytes()
}

err = os.WriteFile("iris_model.tflite", buildTFLite(l1, l2), 0644)
fmt.Println("wrote iris_model.tflite", err)

## Load the flatbuffer with go-tflite and classify

In [ ]:
import "github.com/mattn/go-tflite"

model := tflite.NewModelFromFile("iris_model.tflite")
options := tflite.NewInterpreterOptions()
interpreter := tflite.NewInterpreter(model, options)
interpreter.AllocateTensors()

In [ ]:
correct := 0
for i, f := range x {
    in := interpreter.GetInputTensor(0).Float32s()
    for j, v := range f {
        in[j] = float32(v)
    }
    interpreter.Invoke()
    p := interpreter.GetOutputTensor(0).Float32s()
    best := 0
    for k := range p {
        if p[k] > p[best] {
            best = k
        }
    }
    if best == y[i] {
        correct++
    }
}
fmt.Printf("TFLite inference: %d/%d correct\n", correct, len(x))

In [ ]:
// Classify a new flower: sepal 6.1x2.8cm, petal 4.7x1.2cm.
in := interpreter.GetInputTensor(0).Float32s()
copy(in, []float32{6.1, 2.8, 4.7, 1.2})
interpreter.Invoke()
p := interpreter.GetOutputTensor(0).Float32s()
for i, name := range iris.Names {
    fmt.Printf("%-10s %5.1f%%\n", name, p[i]*100)
}